# Trabajo Fin de Máster  
### Inteligencia artificial explicable para el estudio de la fragmentación urbana  
#### Análisis comparativo del rendimiento y la interpretabilidad de modelos de aprendizaje supervisado para la clasificación de patrones de cerramiento residencial

**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**

> **Autor:** David Fernández Martínez    
> **Email personal:** david.fernxndez.martinez@gmail.com  
> **Email académico:** davidfm8@correo.ugr.es  
> **LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://www.linkedin.com/in/david-fern%C3%A1ndez-mart%C3%ADnez/)  
> **GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Metodología de evaluación de rendimiento y ejecución de experimentos

### 📝 Descripción del notebook

Este *notebook* detalla la metodología aplicada para la evaluación del rendimiento. En primer lugar, se formalizan los modelos seleccionados y las métricas empleadas para analizar su desempeño desde múltiples perspectivas.

A continuación, se presenta el protocolo experimental diseñado para evaluar todos los algoritmos bajo condiciones homogéneas. Tras la ejecución de los experimentos, los resultados generados quedan almacenados en el directorio *output/* y son analizados  en el *notebook* [3.2_Performance_results.ipynb](3.2_Performance_results.ipynb).

### Indice de contenidos
1. [Selección de modelos](#modelos)

2. [Métricas de evaluación de rendimiento](#metricas)
    * [2.1 Métricas globales](#metricas_globales)
    * [2.2 Métricas específicas por clase](#metricas_especificas)

3. [Protocolo experimental de evaluación de rendimiento](#protocolo)
    * [3.1 Preprocesamiento de los datos](#preprocesamiento)
    * [3.2 Implementación práctica](#implementacion)   
    * [3.3 Configuración del aprendizaje sensible al coste](#coste)
    * [3.4 Espacios de búsqueda de hiperparámetros](#hiperparametros)

4. [Ejecución de experimentos](#experimento)

# Configuración de entorno e *imports*

Este proyecto ha sido realizado en un entorno Anaconda con la versión 3.11.15 de *Python*. Las versiones de las librerias requeridas se encuentran en el fichero *requirements-full.txt*.

En esta sección se importan las librerias necesarias para la ejecución de este fichero *jupyter notebook*, se activa el *reload* de módulos externos y se configuran aspectos globales y de reproducibilidad.

In [2]:
# jupyter extensions to automatically reload external modules
%load_ext autoreload
%autoreload 2

In [1]:
import warnings
import random
import numpy as np
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from src.balanced_xgb import BalancedXGBClassifier

# Configuration object
from src.config import cfg

# Performance experiment methods
from src.performance_utils import performance_experiment, performance_experiment_mlflow

**Solución a problemas en *imports***

Si los *imports* del módulo `src` fallan al ejecutar este cuaderno en un entorno diferente, descomente y ejecute la siguiente celda:

```python
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
# Set MLFLOW variable to True if you want to register model evaluation in MLFLOW format
USE_MLFLOW = False

# Set folder to store MLFlow files
mlflow.set_tracking_uri(f"file:{cfg.ROOT_DIR}/mlruns")

In [4]:
# Global configuration
warnings.filterwarnings("ignore")

In [5]:
# Reproducibility
SEED = cfg.SEED 
np.random.seed(SEED)
random.seed(SEED)

<a id="modelos"></a>
# 1. Selección de modelos

La selección de modelos tiene como objetivo cubrir diferentes enfoques de interpretabilidad, así como distintos niveles de complejidad algorítmica, para obtener una visión amplia del compromiso entre interpretabilidad y rendimiento.

Para ello, se consideran dos familias de modelos:

* **Modelos transparentes (intrínsecamente interpretables)**. Algoritmos cuya estructura interna permite comprender de forma directa cómo las variables determinan las predicciones generadas por el modelo.

* **Modelos de caja negra (explicabilidad *post-hoc*)**. Algoritmos cuya complejidad interna impide interpretar directamente el proceso mediante el cual se obtiene una predicción concreta. Su análisis requiere el empleo de técnicas de explicabilidad *post-hoc*, pertenecientes al ámbito de la inteligencia artificial explicable (*Explainable Artificial Intelligence*, XAI), que proporcionan una aproximación al comportamiento del modelo.

En cuanto a los modelos transparentes se han seleccionado dos algoritmos con mecanismos de interpretabilidad intrínseca basados en enfoques diferentes. Por un lado, la **regresión logística multinomial**, cuya interpretabilidad se fundamenta en un modelo aditivo basado en la estimación de coeficientes. Por otro lado, el **árbol de decisión**, que proporciona una explicación de sus predicciones mediante reglas lógicas organizadas de forma secuencial.

Con el objetivo de analizar algoritmos con distintos niveles de complejidad y diferentes capacidades para modelar relaciones no lineales entre las variables, se han seleccionado tres modelos caja de caja negra basados en estrategias de aprendizaje diferentes.

Por un lado, se incluyen las **Máquinas de Vector Soporte (Support Vector Machines, SVM) con kernel de Función de Base Radial (Radial Basis Function, RBF)**. Este algoritmo se fundamenta en la búsqueda de fronteras de decisión que maximicen el margen entre clases, utilizando el kernel RBF para modelar relaciones no lineales mediante una representación implícita en un espacio de mayor dimensionalidad. Su inclusión aporta un enfoque basado en la optimización geométrica de las fronteras de decisión.

Por otro lado, se incorporan dos algoritmos de *ensemble learning*: ***Random Forest*** y ***XGBoost***. Ambos utilizan árboles de decisión como estimadores base, pero emplean estrategias de aprendizaje diferentes. *Random Forest* sigue el paradigma de *bagging*, entrenando múltiples árboles de forma independiente sobre muestras generadas mediante *bootstrap* y combinando posteriormente sus predicciones mediante votación. Por el contrario, *XGBoost* se basa en un enfoque de *gradient boosting*, en el que los árboles se generan de manera secuencial, ajustando cada nuevo modelo para corregir los errores acumulados por los anteriores mediante un proceso de optimización basado en el gradiente.

El fundamento teórico de los modelos seleccionados y sus características de interpretabilidad, distinguiendo entre la interpretabilidad intrínseca de los modelos transparentes y las limitaciones interpretativas de los modelos de caja negra, puede encontrarse con detalle en la sección $4.1$ de la memoria.

<a id="metricas"></a>
# 2. Métricas de evaluación de rendimiento

Para evaluar el rendimiento de los modelos se consideran métricas globales y métricas específicas por clase. Las primeras proporcionan una visión agregada de la capacidad predictiva del modelo, mientras que las segundas permiten analizar su comportamiento individual en cada grado de cerramiento. La combinación de ambos niveles de evaluación permite realizar un análisis más completo, identificando no solo las diferencias globales entre modelos, sino también las categorías de la variable objetivo en las se producen dichas diferencias.

Dado que la validación cruzada anidada (*Nested Cross-Validation*) proporciona $K_{out}$ estimaciones independientes del rendimiento, las métricas globales se calculan en cada partición del bucle externo y se analizan mediante su valor promedio y variabilidad. En cambio, las métricas específicas por clase requieren considerar conjuntamente las predicciones realizadas sobre todos los conjuntos de prueba externos. Para ello, las predicciones obtenidas en cada partición se agregan mediante una estrategia *out-of-fold*. A partir de estas predicciones agregadas se construye una matriz de confusión global, utilizada para calcular las métricas correspondientes a cada clase.

Para definir formalmente las métricas de evaluación se introduce en primer lugar la matriz de confusión generalizada, que extiende la matriz de confusión binaria al contexto de clasificación multiclase.

Sea un problema de clasificación con $K$ categorías mutuamente excluyentes. La matriz de confusión generalizada se define como una matriz cuadrada $C \in \mathbb{N}^{K \times K}$, donde cada elemento $C_{ij}$ representa el número de observaciones cuya clase real corresponde a la categoría $i$ (fila de la matriz) y que han sido predichas por el modelo como pertenecientes a la categoría $j$ (columna de la matriz).

$$
\mathbf{C} = (C_{ij})_{K \times K} = \begin{pmatrix}
C_{11} & C_{12} & \cdots & C_{1K} \\
C_{21} & C_{22} & \cdots & C_{2K} \\
\vdots & \vdots & \ddots & \vdots \\
C_{K1} & C_{K2} & \cdots & C_{KK}
\end{pmatrix}, \quad \text{donde } C_{ij} \text{ con } i, j = 1, \dots, K
$$

A partir de esta matriz, se define para cada clase $k$ (con $k \in {1,\ldots,K}$), los siguientes elementos:

* Verdaderos positivos (*True Positives, $TP_k$*). Corresponden al número de observaciones de la clase $k$ correctamente clasificadas. Estos valores se encuentran en la diagonal principal de la matriz de confusión:
$$
TP_k=C_{kk}
$$
* Falsos positivos (*False Positives, $FP_k$*). Representan las observaciones pertenecientes a otras clases que han sido clasificadas erróneamente como la clase $k$ (Error tipo I). Se obtienen sumando los elementos de la columna $k$, excluyendo el elemento diagonal:
$$
FP_k = \sum_{i=1, i \neq k}^{K} C_{ik}
$$
* Falsos negativos (*False Negatives, $FN_k$*). Representan las observaciones cuya clase real es $k$ y que han sido clasificadas erróneamente en una categoría distinta (Error tipo II). Se obtienen sumando los elementos de la fila $k$, excluyendo el elemento diagonal:
$$
FN_k = \sum_{j=1, j \neq k}^{K} C_{kj}
$$

<a id="metricas_globales"></a>
## 2.1 Métricas globales

Tal y como se ha descrito en el [análisis exploratorio](1_EDA.ipynb), el conjunto de datos presenta desbalanceo de clases. En este contexto, la métrica estándar de *accuracy* puede verse dominada por el comportamiento sobre las clases mayoritarias. Por ello, se han seleccionado métricas que evalúan de forma equilibrada el comportamiento del modelo sobre todas las categorías.

***F1-Score macro***

En este problema no existe una preferencia por reducir los falsos positivos frente a los falsos negativos, ni viceversa, sino que se busca un equilibrio entre la precisión de las predicciones (*Precision*) y la capacidad del modelo para identificar correctamente los complejos residenciales pertenecientes a cada tipología (*Recall*). Por este motivo, el *F1-score* se emplea como métrica principal, ya que combina ambas dimensiones en un único indicador equilibrado.

Dado que todos los grados de cerramiento tienen la misma relevancia en el estudio de la fragmentación urbana, se utiliza la variante *macro*, que calcula el *F1-score* de cada clase de forma independiente y posteriormente obtiene su media aritmética no ponderada. De este modo, todas las categorías contribuyen por igual al resultado final, independientemente de su frecuencia en el conjunto de datos.

Para una clase individual $k$, las métricas *Precision* ($P_k$) y *Recall* ($R_k$) se calculan a partir de los elementos de la matriz de confusión generalizada como:

$$
P_k = \frac{TP_k}{TP_k + FP_k} \quad ; \quad R_k = \frac{TP_k}{TP_k + FN_k}
$$

El *F1-score* de la clase $k$ se define como la media armónica entre ambas métricas:

$$
F_{1,k} = 2 \cdot \frac{P_k \cdot R_k}{P_k + R_k}
$$

Finalmente, el *F1-score Macro* se obtiene calculando la media aritmética no ponderada de los valores de *F1-score* correspondientes a las $K$ clases:

$$
F_1\text{-Score Macro} = \frac{1}{K} \sum_{k=1}^{K} F_{1,k}
$$

La métrica se ha implementado mediante el método *f1_score* del módulo *metrics* de *scikit-learn*, utilizando la configuración *average='macro'*. Además, se ha empleado como métrica objetivo para la selección de hiperparámetros durante la validación cruzada interna del esquema de *Nested Cross-Validation*.

**Coeficiente de Correlación de Matthews *MCC***

El coeficiente de correlación de Matthews (*MCC*) se incorpora como métrica complementaria al *F1-score macro*, proporcionando una visión alternativa del rendimiento del modelo en presencia de desbalanceo de clases. A diferencia del *F1-score macro*, que evalúa el equilibrio del rendimiento entre clases, el *MCC* considera conjuntamente todos los elementos de la matriz de confusión, proporcionando una medida global de la calidad de la clasificación.

Esta métrica evalúa el grado de concordancia entre las etiquetas reales y las predicciones, interpretando ambos conjuntos como variables categóricas cuya correlación se desea cuantificar. En su formulación para problemas multiclase, el *MCC* se define a partir de la matriz de confusión de la siguiente forma:

$$
MCC = \frac{
c \cdot s - \sum_{k}^{K} p_k \cdot t_k
}{\sqrt{
(s^2 - \sum_{k}^{K} p_k^2) \cdot
(s^2 - \sum_{k}^{K} t_k^2)
}}
$$

donde:

* $t_k=\sum_{j}^{K} C_{kj}$ representa la frecuencia real de la clase $k$ (suma de la fila $k$ en la matriz de confusión generalizada).

* $p_k=\sum_{i}^{K} C_{ik}$ representa el total de predicciones asignadas a la clase $k$ (suma de la columna $k$ en la matriz de confusión generalizada).

* $c=\sum_{k}^{K} C_{kk}$ es el número de muestras predichas correctamente.

* $s=\sum_{i}^{K} \sum_{j}^{K} C_{ij}$ es el número total de muestras.

El numerador cuantifica la divergencia entre el rendimiento observado y el rendimiento atribuible al azar considerando la distribución real de las clases. El denominador normaliza esta relación teniendo en cuenta la variabilidad de las clases reales y predichas. El coeficiente *MCC* toma valores entre $-1$ y $1$, donde $1$ indica una clasificación perfecta, $0$ un rendimiento equivalente al azar y los valores negativos $^1$ una correlación inversa entre las predicciones y las clases reales.

Esta métrica se ha implementado mediante el método *matthews_corrcoef* del módulo *metrics* de *scikit-learn*.

> $^1$ En problemas multiclase el límite inferior del *MCC* depende del número de categorías y del grado de desbalanceo.

***accuracy***

El *accuracy* se incluye entre las métricas reportadas debido a su amplia utilización en problemas de clasificación, lo que facilita la comparación de los resultados con futuros estudios que empleen este conjunto de datos. No obstante, debido al desbalanceo de clases, la evaluación del rendimiento se basa en la métricas *F1-Score Macro* y *MCC*.

El *accuracy* representa la proporción total de aciertos sobre el total de la muestra ($N$). Se calcula sumando la diagonal de la matriz de confusión generalizada y dividiendo entre la suma de todos sus elementos:

$$
\text{Accuracy} = \frac{\sum_{k=1}^{K} C_{kk}}{\sum_{i=1}^{K} \sum_{j=1}^{K} C_{ij}} = \frac{\sum_{k=1}^{K} TP_k}{N}
$$

Esta métrica se ha implementado mediante el método *accuracy_score()* del paquete *metrics* de *scikit-learn*.


<a id="metricas_especificas"></a>
## 2.2 Métricas específicas por clase

Con el objetivo de analizar de forma desagregada el comportamiento de los modelos sobre cada uno de los cinco grados de cerramiento, se calculan métricas específicas para cada clase.

Para ello, se emplea una estrategia *out-of-fold* basada en las predicciones obtenidas durante el bucle externo de la validación cruzada anidada. Dado que los conjuntos de prueba externos (*Outer Test Set*) son mutuamente excluyentes, al finalizar el procedimiento se dispone de una única predicción para cada uno de los $642$ complejos residenciales del conjunto de datos, generada por un modelo que no ha utilizado la observación correspondiente durante su entrenamiento.

A partir de estas predicciones y sus etiquetas reales se construye una matriz de confusión generalizada que permite calcular la *Precision*, el *Recall* y el *F1-score* asociados a cada clase de la variable objetivo, siguiendo las formulaciones definidas en las Ecuaciones correspondientes.

<a id="protocolo"></a>
# 3. Protocolo experimental de evaluación de rendimiento

Con el objetivo de realizar una comparación rigurosa entre los modelos de clasificación considerados, se ha definido un protocolo experimental común que garantiza que todos los algoritmos sean evaluados bajo las mismas condiciones. De este modo, las diferencias observadas en las métricas de rendimiento pueden atribuirse al comportamiento de los propios modelos y no a variaciones en el procedimiento de evaluación.

En esta sección se describe la implementación práctica del protocolo de evaluación de rendimiento así como su configuración experimental. En primer lugar, se justifica la ausencia de etapas de preprocesamiento y se presenta el flujo de ejecución desarrollado para aplicar la estrategia de [validación cruzada anidada](2_Folds_generation.ipynb) y recopilar las métricas de rendimiento. Posteriormente, se describen las configuraciones específicas incorporadas para abordar el desbalanceo de clases mediante mecanismos de aprendizaje sensible al coste (*cost-sensitive learning*), así como los espacios de búsqueda de hiperparámetros utilizados para la optimización de cada algoritmo.

<a id="preprocesamiento"></a>
## 3.1 Preprocesamiento de los datos

El conjunto de datos está compuesto exclusivamente por variables predictoras categórico binarias codificadas mediante valores $0$ y $1$. Por este motivo, no se han aplicado transformaciones adicionales como escalado, normalización o técnicas de codificación categórica, dado que la representación utilizada es directamente compatible con todos los algoritmos evaluados.

<a id="implementacion"></a>
## 3.2 Implementación práctica

La implementación práctica del protocolo experimental de evaluación de rendimiento se ha desarrollado utilizando la biblioteca *scikit-learn* de *Python* como base para la implementación de los algoritmos de clasificación, las métricas de evaluación, la generación de particiones estratificadas y el proceso de optimización de hiperparámetros mediante validación cruzada.

El flujo completo de ejecución se encuentra encapsulado en la función *performance_experiment()*, ubicada en el módulo *src/performance_utils.py*. La siguiente figura representa mediante un diagrama de bloques el procedimiento implementado en esta función, que se resumen en los siguientes pasos:

1. Carga del conjunto de datos y de las particiones predefinidas correspondientes a la validación cruzada externa.

2. Para cada partición del bucle externo:

   * Construcción de los conjuntos *Outer Train Set* y *Outer Test Set*.
   * Carga de las particiones internas asociadas al *Outer Train Set*.
   * Optimización de hiperparámetros mediante *GridSearchCV*, utilizando exclusivamente las particiones definidas para la validación cruzada interna.
   * Reentrenamiento del modelo, con la configuración óptima obtenida, sobre todo el conjunto *Outer Train Set*.
   * Evaluación del modelo sobre el conjunto independiente *Outer Test Set*.
   * Cálculo de las métricas globales (*F1-score macro*, *MCC* y *accuracy*).
   * Almacenamiento de las predicciones y etiquetas reales correspondientes al conjunto *Outer Test Set* para la posterior construcción de las métricas específicas por clase.

3. Almacenamiento de los resultados de las métricas globales obtenidas en cada partición externa.

4. Construcción de la matriz de confusión generalizada a partir de las predicciones *out-of-fold* y cálculo de las métricas específicas por clase.

5. Almacenamiento de la matriz de confusión generalizada y de las métricas específicas por clase.

<img src="../images/experiment/performance_experiment.png" alt="performance_experiment()" width="80%">

Además de la función *performance_experiment()*, se ha desarrollado la función adicional *performance_experiment_mlflow()*, que implementa el mismo protocolo experimental incorporando el registro estructurado de métricas, parámetros y artefactos mediante *MLflow*. Esta versión facilita la trazabilidad y reproducibilidad de los experimentos, permitiendo que futuros usuarios del repositorio puedan integrar el protocolo experimental de este trabajo con esta plataforma si lo requieren.

In [6]:
# Select method according to USE_MLFLOW parameter
if USE_MLFLOW:
    experiment_method = performance_experiment_mlflow
else:
    experiment_method = performance_experiment

<a id="coste"></a>
## 3.3 Configuración del aprendizaje sensible al coste

Para que un modelo de clasificación sea adecuado en este problema, debe proporcionar un rendimiento equilibrado para todas las clases, ya que cada grado de cerramiento constituye una categoría de interés propio dentro del estudio.

El desequilibrio de clases presente en la variable objetivo, puede provocar que los algoritmos favorezcan los grados de cerramiento mayoritarios, reduciendo su capacidad para identificar correctamente los menos representados. Para abordar esta limitación, se han empleado mecanismos nativos de aprendizaje sensible al coste (*cost-sensitive learning*) durante la fase de entrenamiento de cada modelo.

Bajo este enfoque, el protocolo experimental evalúa cada algoritmo considerando conjuntamente su arquitectura de aprendizaje y su mecanismo intrínseco de aprendizaje sensible al coste. De este modo, se dota a cada modelo de las herramientas disponibles en su propia implementación para hacer frente al desequilibrio estructural de los datos, realizando así una evaluación representativa de su comportamiento en un escenario de aplicación real.

La implementación del aprendizaje sensible al coste se ha realizado mediante el parámetro *class_weight="balanced"* disponible en los algoritmos implementados a través de *scikit-learn*. Esta configuración asigna automáticamente un peso a cada clase de forma inversamente proporcional a su frecuencia en el conjunto de entrenamiento, otorgando una mayor relevancia a las clases minoritarias durante el proceso de aprendizaje.

En los modelos basados en optimización, como la regresión logística multinomial y *SVM*, estos pesos modifican la función objetivo, aumentando la penalización asociada a los errores cometidos sobre las clases menos representadas. En los modelos basados en árboles, como el árbol de decisión y *Random Forest*, los pesos se incorporan en el cálculo de las medidas de impureza utilizadas durante la generación de particiones, favoreciendo divisiones que consideren adecuadamente la distribución de todas las clases.

El modelo *XGBoost* se ha implementado mediante la librería *xgboost* desarrollada por los propios autores del algoritmo. A diferencia de los estimadores disponibles en *scikit-learn*, esta librería no incorpora un parámetro equivalente a *class_weight="balanced"* en problemas de clasificación multiclase. Para incorporar este mecanismo, garantizando compatibilidad con el ecosistema de *scikit-learn*, se ha desarrollado un contenedor (*wrapper*) sobre el estimador *XGBClassifier* que permite incorporar de forma explícita el vector de pesos durante el entrenamiento.

En cada llamada al método *fit*, el contenedor calcula el vector de pesos mediante la función *compute_sample_weight()* del módulo *utils.class_weights* de *scikit-learn* y lo transmite al estimador original de *XGBoost*. De esta forma, el modelo incorpora un aprendizaje sensible al coste equivalente al aplicado en el resto de algoritmos, asignando una mayor influencia a las muestras pertenecientes a clases menos representadas.

El cálculo dinámico de los pesos en cada ejecución de *fit* garantiza que estos se estimen exclusivamente a partir de las muestras utilizadas para entrenar el modelo en dicha ejecución. Esto resulta especialmente relevante en el esquema *Nested Cross Validation*, donde los pesos deben recalcularse para cada partición utilizando únicamente el conjunto de entrenamiento correspondiente, evitando introducir información del conjunto de validación durante la optimización de hiperparámetros o del conjunto de prueba durante la estimación del rendimiento.

Esta solución se ha encapsulado en la clase *BalancedXGBClassifier* ubicada en el módulo *src/balanced_xgb.py*. Esta clase hereda la interfaz estándar de *scikit-learn* para garantizar compatibilidad con el ecosistema del protocolo experimental.

<a id="hiperparametros"></a>
## 3.4 Espacios de búsqueda de hiperparámetros

En la siguiente tabla se recoge el espacio de búsqueda de hiperparámetros definido para cada modelo, junto con el coste computacional expresado como número de entrenamientos requeridos ($N_t$). En el esquema *Nested Cross-Validation* con una configuración $K_{out}=K_{in}=5$, el número total de entrenamientos $N_t$ para un modelo con $H$ configuraciones de hiperparámetros viene dado por:
$$
N_t = K_{out}\cdot(K_{in}\cdot H+1)=5\cdot(5\cdot H+1)
$$
donde el término $K_{out}\cdot K_{in}\cdot H$ corresponde a los entrenamiento realizados durante la búsqueda de hiperparámetros en el bucle interno, mientras que el término $K_{out}$ representa el reentrenamiento del modelo óptimo sobre el *Outer Train Set* de la partición externa.

<table>
  <thead>
    <tr>
      <th style="text-align:left">Modelo</th>
      <th style="text-align:left">Hiperparámetro</th>
      <th style="text-align:left">Espacio de búsqueda</th>
      <th style="text-align:center"><b><i>N<sub>t</sub></i></b></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Regresión logística multinomial</td>
      <td><i>Regularización inversa (C)</i></td>
      <td>[10, 100, 1000, 10000]</td>
      <td style="text-align:center">105</td>
    </tr>
    <tr>
      <td rowspan="4">Árbol de decisión</td>
      <td><i>max_depth</i></td>
      <td>6</td>
      <td rowspan="4" style="text-align:center">755</td>
    </tr>
    <tr>
      <td><i>max_leaf_nodes</i></td>
      <td>[10, 15, 20, 25, 30]</td>
    </tr>
    <tr>
      <td><i>min_samples_leaf</i></td>
      <td>[5, 10, 15]</td>
    </tr>
    <tr>
      <td><i>criterion</i></td>
      <td>["gini", "entropy"]</td>
    </tr>
    <tr>
      <td rowspan="2">SVM con kernel RBF</td>
      <td><i>Regularización (C)</i></td>
      <td>[0.1, 1, 10, 100]</td>
      <td rowspan="2" style="text-align:center">405</td>
    </tr>
    <tr>
      <td><i>Kernel (&gamma;)</i></td>
      <td>["scale", "auto", 0.01, 0.1]</td>
    </tr>
    <tr>
      <td rowspan="5"><i>Random Forest</i></td>
      <td><i>n_estimators</i></td>
      <td>[100, 200, 300]</td>
      <td rowspan="5" style="text-align:center">2705</td>
    </tr>
    <tr>
      <td><i>max_depth</i></td>
      <td>[5, 10, None]</td>
    </tr>
    <tr>
      <td><i>max_features</i></td>
      <td>["sqrt", 0.3, 0.4]</td>
    </tr>
    <tr>
      <td><i>criterion</i></td>
      <td>["gini", "entropy"]</td>
    </tr>
    <tr>
      <td><i>min_samples_split</i></td>
      <td>[2, 5]</td>
    </tr>
    <tr>
      <td rowspan="7"><i>XGBoost</i></td>
      <td><i>n_estimators</i></td>
      <td>[100, 300]</td>
      <td rowspan="7" style="text-align:center">3205</td>
    </tr>
    <tr>
      <td><i>learning_rate</i></td>
      <td>[0.01, 0.1]</td>
    </tr>
    <tr>
      <td><i>max_depth</i></td>
      <td>[5, 10]</td>
    </tr>
    <tr>
      <td><i>subsample</i></td>
      <td>[0.8, 1]</td>
    </tr>
    <tr>
      <td><i>colsample_bytree</i></td>
      <td>[0.8, 1]</td>
    </tr>
    <tr>
      <td><i>Regularización L2 (&lambda;)</i></td>
      <td>[1, 5]</td>
    </tr>
    <tr>
      <td>Umbral de ganancia (&gamma;)</td>
      <td>[0, 0.3]</td>
    </tr>
  </tbody>
</table>

Las decisiones metodológicas referentes a la configuración del espacio de hiperparámetros se explican en detalle en la sección 4.2.3.4 de la memoria.

<a id="experimento"></a>
# 4. Ejecución de experimentos

En esta sección se ejecuta el protocolo experimental desarrollado para cada uno de los modelos seleccionados. Los resultados se almacenan en el directorio:

*output/$K_{out}$ x $K_{in}$ _NCV/model_name*

donde $K_{out}$ y $K_{in}$ son el número de particiones externa e interna, dadas por las variables `OUTER_SPLITS` e `INNER_SPLITS` de *src/config.py* y *model_name* es el nombre del modelo que se le pasa a la función en el parámetro experiment_name.

In [7]:
################################
# Multinomial Logistic Regression
################################

LR_pipeline = Pipeline([
    ("model", LogisticRegression(
        class_weight="balanced",
        solver="lbfgs",
        penalty="l2",
        random_state = SEED
    ))
])

LR_param_grid = {
    "model__C": [10, 100, 1000, 10000],
}

LR_global_results_df, LR_cm_df, LR_class_report_df = experiment_method(
    config=cfg,
    pipeline=LR_pipeline,
    param_grid=LR_param_grid,
    experiment_name="Logistic_Regression"
)


NESTED CROSS-VALIDATION EXPERIMENT
Experiment Name      : Logistic_Regression
Outer CV Folds       : 5
Inner CV Folds       : 5

Hyperparameter Grid:
model__C                 : [10, 100, 1000, 10000]

--------------------------------------------------------------------------------
OUTER FOLD [1/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (5 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

--------------------------------------------------------------------------------
OUTER FOLD [2/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (5 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

------------------------------------------------

In [8]:
################################
# Decision Tree
################################

DT_pipeline = Pipeline([
    ("model", DecisionTreeClassifier(
        class_weight = "balanced",
        max_depth = 6,
        random_state = SEED
    ))
])

DT_param_grid = {
    "model__max_leaf_nodes": [10, 15, 20, 25, 30],
    "model__min_samples_leaf": [5, 10, 15],
    "model__criterion": ["gini", "entropy"],
}



DT_global_results_df, DT_cm_df, DT_class_report_df = experiment_method(
    config=cfg,
    pipeline=DT_pipeline,
    param_grid=DT_param_grid,
    experiment_name="Decision_Tree"
)


NESTED CROSS-VALIDATION EXPERIMENT
Experiment Name      : Decision_Tree
Outer CV Folds       : 5
Inner CV Folds       : 5

Hyperparameter Grid:
model__max_leaf_nodes    : [10, 15, 20, 25, 30]
model__min_samples_leaf  : [5, 10, 15]
model__criterion         : ['gini', 'entropy']

--------------------------------------------------------------------------------
OUTER FOLD [1/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (5 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

--------------------------------------------------------------------------------
OUTER FOLD [2/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (5 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 5 folds for each of 3

In [9]:
################################
# SVM With RBF Kernel
################################

SVM_pipeline = Pipeline([
    ("model", SVC(
        kernel = "rbf",
        decision_function_shape = 'ovr',
        class_weight = "balanced",
        random_state = SEED
    ))
])

SVM_param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", "auto", 0.01, 0.1]
}



SVM_global_results_df, SVM_cm_df, SVM_class_report_df = experiment_method(
    config=cfg,
    pipeline=SVM_pipeline,
    param_grid=SVM_param_grid,
    experiment_name="SVM"
)


NESTED CROSS-VALIDATION EXPERIMENT
Experiment Name      : SVM
Outer CV Folds       : 5
Inner CV Folds       : 5

Hyperparameter Grid:
model__C                 : [0.1, 1, 10, 100]
model__gamma             : ['scale', 'auto', 0.01, 0.1]

--------------------------------------------------------------------------------
OUTER FOLD [1/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (5 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

--------------------------------------------------------------------------------
OUTER FOLD [2/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (5 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

-----------

In [10]:
################################
# Random Forest
################################

RF_pipeline = Pipeline([
    ("model", RandomForestClassifier(
        class_weight = "balanced_subsample",
        random_state = SEED
    ))
])

RF_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, None],
    "model__max_features": ["sqrt", 0.3, 0.4],
    "model__criterion": ['gini', 'entropy'],
    "model__min_samples_split": [2, 5],
}


RF_global_results_df, RF_cm_df, RF_class_report_df = experiment_method(
    config=cfg,
    pipeline=RF_pipeline,
    param_grid=RF_param_grid,
    experiment_name="Random_Forest"
)


NESTED CROSS-VALIDATION EXPERIMENT
Experiment Name      : Random_Forest
Outer CV Folds       : 5
Inner CV Folds       : 5

Hyperparameter Grid:
model__n_estimators      : [100, 200, 300]
model__max_depth         : [5, 10, None]
model__max_features      : ['sqrt', 0.3, 0.4]
model__criterion         : ['gini', 'entropy']
model__min_samples_split : [2, 5]

--------------------------------------------------------------------------------
OUTER FOLD [1/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (5 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 5 folds for each of 108 candidates, totalling 540 fits

--------------------------------------------------------------------------------
OUTER FOLD [2/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (5 fold

In [11]:
################################
# XGBoost
################################

XG_pipeline = Pipeline([
    ("model", BalancedXGBClassifier(
        random_state = SEED,
        sampling_method = "uniform",
        objective= "multi:softmax",
        eval_metric="mlogloss",
        ))
])

XG_param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [5, 10],
    "model__learning_rate": [0.01, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__gamma": [0, 0.3],
    "model__reg_lambda": [1, 5]
}


XG_global_results_df, XG_cm_df, XG_class_report_df = experiment_method(
    config=cfg,
    pipeline=XG_pipeline,
    param_grid=XG_param_grid,
    experiment_name="XGBoost"
)


NESTED CROSS-VALIDATION EXPERIMENT
Experiment Name      : XGBoost
Outer CV Folds       : 5
Inner CV Folds       : 5

Hyperparameter Grid:
model__n_estimators      : [100, 300]
model__max_depth         : [5, 10]
model__learning_rate     : [0.01, 0.1]
model__subsample         : [0.8, 1.0]
model__colsample_bytree  : [0.8, 1.0]
model__gamma             : [0, 0.3]
model__reg_lambda        : [1, 5]

--------------------------------------------------------------------------------
OUTER FOLD [1/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (5 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 5 folds for each of 128 candidates, totalling 640 fits

--------------------------------------------------------------------------------
OUTER FOLD [2/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
In